In [1]:
import os
import re
import unidecode
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, FloatType, TimestampType

# --- Função de Processamento para cada Arquivo ---
# Esta função será distribuída e executada em paralelo pelo Spark para cada arquivo.
def processar_conteudo_arquivo(file_info):
    """
    Processa o conteúdo de um único arquivo CSV, extraindo metadados do cabeçalho
    e os dados principais.
    
    Args:
        file_info (tuple): Uma tupla contendo (caminho_do_arquivo, conteudo_completo_do_arquivo).

    Returns:
        list: Uma lista de dicionários, onde cada dicionário representa uma linha de dados
              enriquecida com os metadados do arquivo.
    """
    caminho_arquivo, conteudo_completo = file_info
    linhas = conteudo_completo.split('\n')
    
    # 1. Extração de Metadados do Cabeçalho (primeiras 8 linhas)
    metadados = {}
    cabecalho_linhas = linhas[:8]
    for linha in cabecalho_linhas:
        if ';' in linha:
            partes = linha.split(';', 1)
            chave = partes[0].strip()
            valor = partes[1].strip() if len(partes) > 1 else None
            
            # Limpeza e normalização da chave
            chave_limpa = unidecode.unidecode(chave).upper()
            chave_limpa = re.sub(r'[^A-Z0-9 ]', '', chave_limpa)
            if 'REGIO' in chave_limpa: chave_limpa = 'REGIAO'
            if 'ESTACO' in chave_limpa: chave_limpa = 'ESTACAO'
            
            metadados[chave_limpa] = valor

    # 2. Extração de Dados Tabulares (a partir da linha 9)
    registros = []
    dados_linhas = linhas[9:] # A linha 8 é o header dos dados
    
    for linha_dado in dados_linhas:
        valores = linha_dado.strip().split(';')
        # Garante que a linha tem o número mínimo de colunas para evitar erros
        if len(valores) >= 2 and valores[0]:
            try:
                # O nome da coluna de data pode variar
                data = valores[0]
                # O nome da coluna de temperatura é mais consistente
                temperatura = valores[10] # Baseado no formato de dados do INMET
                
                # Monta um dicionário para a linha de dados
                registro = {
                    'Regiao': metadados.get('REGIAO'),
                    'UF': metadados.get('UF'),
                    'Estacao': metadados.get('ESTACAO'),
                    'Latitude': float(str(metadados.get('LATITUDE', '0')).replace(',', '.')),
                    'Longitude': float(str(metadados.get('LONGITUDE', '0')).replace(',', '.')),
                    'Altitude': float(str(metadados.get('ALTITUDE', '0')).replace(',', '.')),
                    'DataOriginal': data,
                    'TemperaturaOriginal': temperatura
                }
                registros.append(registro)
            except (IndexError, ValueError):
                # Ignora linhas malformadas
                pass
                
    return registros

# --- Script Principal PySpark ---
def main():
    spark = SparkSession.builder \
        .appName("Processamento INMET com PySpark") \
        .master("spark://spark-master:7077") \
        .getOrCreate()

    # Ajuste o caminho para ser mais flexível, se necessário
    caminho_base = "../data/bronze/INMET/*/*.CSV"
    pasta_destino = "../data/bronze/INMET_PARQUET"
    caminho_leitura = f"{caminho_base}" # Lê de todos os anos

    rdd_arquivos = spark.sparkContext.wholeTextFiles(caminho_leitura)
    rdd_processado = rdd_arquivos.flatMap(processar_conteudo_arquivo)
    
    # Persist para evitar recomputação
    rdd_processado.persist()

    if rdd_processado.isEmpty():
        print("Nenhum dado foi processado. Verifique o caminho e o conteúdo dos arquivos.")
        spark.stop()
        return

    # --- SOLUÇÃO AQUI: DEFINIR O SCHEMA EXPLÍCITO ---
    # Os nomes dos campos (name) devem ser EXATAMENTE IGUAIS às chaves do dicionário
    # que você cria na função processar_conteudo_arquivo.
    schema_definido = StructType([
        StructField("Regiao", StringType(), True),
        StructField("UF", StringType(), True),
        StructField("Estacao", StringType(), True),
        StructField("Latitude", FloatType(), True),
        StructField("Longitude", FloatType(), True),
        StructField("Altitude", FloatType(), True),
        StructField("DataOriginal", StringType(), True),
        StructField("TemperaturaOriginal", StringType(), True)
    ])

    # Agora, crie o DataFrame FORNECENDO o schema. O Spark não precisará mais adivinhar.
    df = spark.createDataFrame(rdd_processado, schema=schema_definido)

    # O resto do seu código de transformação funciona da mesma forma
    df_transformado = df \
        .withColumn('Temperatura', F.regexp_replace(F.col('TemperaturaOriginal'), ',', '.').cast(FloatType())) \
        .withColumn('Data', F.to_timestamp(F.col('DataOriginal'), 'yyyy-MM-dd')) \
        .filter(F.col('Temperatura') != -9999.0) \
        .withColumn('Tempo', F.date_format(F.col('Data'), 'HH:mm:ss')) \
        .withColumn('Ano', F.year(F.col('Data'))) \
        .select(
            'Data', 'Tempo', 'Temperatura', 'Regiao', 'UF', 'Estacao',
            'Latitude', 'Longitude', 'Altitude', 'Ano'
        )
    
    print(f"Salvando os dados transformados em {pasta_destino}...")
    df_transformado.write \
        .partitionBy("Ano") \
        .mode("overwrite") \
        .parquet(pasta_destino)

    print("Processamento concluído com sucesso!")
    
    spark.stop() #Mudar isso, e se der erro no meio do processo ? Ele precisa encerrar sozinha.

if __name__ == '__main__':
    main()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/03 01:13:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/10/03 01:32:39 WARN TaskSetManager: Stage 0 contains a task of very large size (1220 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Salvando os dados transformados em ../data/silver/INMET_PARQUET...


25/10/03 01:37:37 WARN TaskSetManager: Stage 1 contains a task of very large size (1220 KiB). The maximum recommended task size is 1000 KiB.
25/10/03 01:37:40 WARN TaskSetManager: Lost task 0.0 in stage 1.0 (TID 1) (172.18.0.5 executor 0): org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value 'TEMPERATURA M�NIMA NA HORA ANT. (AUT) (�C)' of the type "STRING" cannot be cast to "FLOAT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
line 116 in cell [1]

	at org.apache.spark.sql.errors.QueryExecutionErrors$.invalidInputInCastToNumberError(QueryExecutionErrors.scala:145)
	at org.apache.spark.sql.errors.QueryExecutionErrors.invalidInputInCastToNumberError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.sort_addToSorter_0$(Unknown Sour

NumberFormatException: [CAST_INVALID_INPUT] The value 'TEMPERATURA M�NIMA NA HORA ANT. (AUT) (�C)' of the type "STRING" cannot be cast to "FLOAT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
line 116 in cell [1]
